# Unit 9 exercises: build a line, follow it, and measure it honestly

A racing line is a set of offsets along the track's normals, made from a few control values of a cubic B-spline and chosen to be as smooth as the road allows.
These exercises build the pieces of the lesson's solver one at a time, check them on a circle where the answers are known, solve a line on a new track, follow it with the line-relative clean rule, and compare it with the centreline at the same f.
The Challenges try two variations the lesson did not: a late-apex window, and a narrower margin.
Every number here is a number about racekit's **carla-like car**: a model, not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.
Every race uses `carla_like(noise=True)` with the stated seeds; a `test-circuit` race takes a few seconds and a `monza-like` race about twice that, and the exercises keep the number of races small.
A line solve takes a few seconds on a `test-circuit` track and up to about 10 s on `monza-like`.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_08, UNIT_09 = AGENTS / "unit-08.py", AGENTS / "unit-09.py"
workdir = Path(mkdtemp(prefix="roar-unit-09-ex-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the cross-track error from the race's own line
REACHED_RADIUS = 3.0  # m: the agents' "waypoint reached" radius

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
# (monza-like's section 0 never slid in its probe, so its number is a lower bound; the planner uses it as measured.)
MEASURED_MU_G = {
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "monza-like": (5.38, 8.96, 9.12, 9.14, 9.70, 7.97),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def menger(x, y):
    # Unit 2's signed Menger curvature at every point of a closed polyline (NumPy arrays; left turns positive).
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def line_summary(x, y):
    # (total squared curvature sum kappa^2 ds, lap length in m, tightest radius in m) of a closed line.
    kappa = menger(x, y)
    ds = np.hypot(np.roll(x, -1) - x, np.roll(y, -1) - y)
    return float(np.sum(kappa ** 2 * ds)), float(ds.sum()), float(1.0 / np.max(np.abs(kappa)))

## Exercise 1 - The B-spline's four weights

Waypoint i of an n-waypoint lap with m = n // step control values sits at t = i·m / n; with j = ⌊t⌋ and u = t − j, its offset is w_0 c_{j−1} + w_1 c_j + w_2 c_{j+1} + w_3 c_{j+2} (indices mod m), where

w_0 = (1 − u)³ / 6, w_1 = (3u³ − 6u² + 4) / 6, w_2 = (−3u³ + 3u² + 3u + 1) / 6, w_3 = u³ / 6.

Write:

- `spline_weights(u)`: returns the tuple (w_0, w_1, w_2, w_3);
- `spline_basis(n, step=10)`: returns an n × m NumPy array whose entry [i, k] is the weight of control value k in waypoint i's offset (add each of the four weights into column (j − 1 + k) mod m, k = 0 … 3).

Define:

- `spline_weights`, `spline_basis`: the functions;
- `weights_at`: a dict from u to `spline_weights(u)` for u = 0.0, 0.25, 0.5 and 0.75;
- `smallest_weight`: the smallest of the four weights over the 101 values `np.linspace(0.0, 1.0, 101)`;
- `worst_sum_error`: the largest |w_0 + w_1 + w_2 + w_3 − 1| over the same 101 values;
- `toy_basis`: `spline_basis(40, 10)` (so m = 4);
- `nonzeros_per_row`: the sorted list of the distinct numbers of entries > 0 in a row of `toy_basis`;
- `toy_control`: the NumPy array (2.0, −1.0, 3.0, 0.5), and `toy_offsets = toy_basis @ toy_control`;
- `bound_holds`: `True` when every |toy_offsets[i]| is at most the largest |toy_control[k]| over the columns k where row i of `toy_basis` is > 0;
- `why_bounds_carry`: one or two sentences: why does a limit |c_k| ≤ h on every control value guarantee |n_i| ≤ h at every waypoint, and which two properties of the weights does that need?

In [ ]:
def spline_weights(u):
    return None  # TODO: (w0, w1, w2, w3)


def spline_basis(n, step=10):
    return None  # TODO: an n x (n // step) NumPy array


weights_at = {}  # TODO
smallest_weight = None  # TODO
worst_sum_error = None  # TODO
toy_basis = None  # TODO
nonzeros_per_row = []  # TODO
toy_control, toy_offsets = None, None  # TODO
bound_holds = None  # TODO
why_bounds_carry = "TODO"

## Exercise 2 - Normals and second differences on a circle

On a circle the answers are known, so it is a good place to check the tools.
Make a circle of radius 50 m around the origin, 300 points anticlockwise: `theta = 2 * np.pi * np.arange(300) / 300`, `cx = 50 * np.cos(theta)`, `cy = 50 * np.sin(theta)`.

Write:

- `normals(x, y)`: the left unit normal at every point of a closed polyline, as two NumPy arrays (nx, ny): the chord from point i − 1 to point i + 1 is (d_x, d_y) = (x_{i+1} − x_{i−1}, y_{i+1} − y_{i−1}) with the ends wrapping round, h = √(d_x² + d_y²), and the normal is (−d_y / h, d_x / h);
- `second_difference(values)`: v_{i+1} − 2 v_i + v_{i−1} for every i, the ends wrapping round, along the **first axis** (so a 1-D array and every column of a 2-D array both work; `np.roll(values, -1, axis=0)` is v_{i+1}).

Define:

- `normals`, `second_difference`: the functions;
- `cnx`, `cny`: the circle's normals;
- `points_inward`: `True` when every normal points at the centre, that is, (cnx, cny) is within 1e-9 of (−cx / 50, −cy / 50) at every point;
- `inner_x`, `inner_y`: the circle's points moved n = 3.0 m along their normals (the same offset everywhere);
- `inner_radius`: the mean distance of the moved points from the origin;
- `inner_curvature`: the mean Menger curvature of the moved points (the setup cell's `menger`);
- `second_difference_length`: the mean length √((Δ²x)² + (Δ²y)²) of the circle's (unmoved) second differences;
- `kappa_ds_squared`: κ·Δs² for the circle, with κ = 1 / 50 and Δs the mean distance between neighbouring points;
- `relative_gap`: |`second_difference_length` − `kappa_ds_squared`| / `kappa_ds_squared`;
- `what_second_differences_measure`: one sentence: what does a second difference of a line's points measure, and why is it only a stand-in for curvature on a real track whose points are not evenly spaced?

In [ ]:
def normals(x, y):
    return None, None  # TODO: (nx, ny)


def second_difference(values):
    return None  # TODO


theta = 2 * np.pi * np.arange(300) / 300
cx, cy = 50 * np.cos(theta), 50 * np.sin(theta)
cnx, cny = None, None  # TODO
points_inward = None  # TODO
inner_x, inner_y = None, None  # TODO
inner_radius = None  # TODO
inner_curvature = None  # TODO
second_difference_length = None  # TODO
kappa_ds_squared = None  # TODO
relative_gap = None  # TODO
what_second_differences_measure = "TODO"

## Exercise 3 - The smoothness objective as a matrix

For a lap with points (x, y), left normals (N_x, N_y) and a basis B (n × m), the line with control values c has points x' = x + N_x (B c), y' = y + N_y (B c), and the smoothness objective Σ (Δ²x')² + (Δ²y')² equals ‖A c − b‖² with

- A = the rows of `second_difference(nx[:, None] * B)` stacked **above** the rows of `second_difference(ny[:, None] * B)` (2n × m), and
- b = −(the n entries of `second_difference(x)` followed by the n entries of `second_difference(y)`).

Write `smoothness_system(x, y, basis)`, returning `(A, b)`.

Use `test-circuit` (`track = load_track("test-circuit")`, its `track.x`, `track.y`) with `spline_basis(n, 10)`.
The reference's control values are a module constant of `agents/unit-09.py`; read them like this:

```python
with shim_installed():
    Unit09, module_name = load_submission(UNIT_09)
shipped = np.array(sys.modules[module_name].LINE_CONTROL)
```

Define:

- `smoothness_system`: the function;
- `A`, `b`: its result on `test-circuit`;
- `objective_centre`: ‖A c − b‖² for c = 0 (the centreline), and `direct_centre`: Σ (Δ²x)² + (Δ²y)² computed directly from the centreline's points;
- `objective_line`: ‖A c − b‖² for c = `shipped`, and `direct_line`: Σ (Δ²x')² + (Δ²y')² computed directly from the line's points;
- `largest_gap`: the larger of |`objective_centre` − `direct_centre`| and |`objective_line` − `direct_line`|;
- `objective_reduction`: 1 − `objective_line` / `objective_centre`;
- `curvature_reduction`: 1 − (the line's Σκ²Δs) / (the centreline's Σκ²Δs), from the setup cell's `line_summary`;
- `same_story`: one sentence: do the stand-in and the curvature agree on the direction of the change, and on its size?

In [ ]:
def smoothness_system(x, y, basis):
    return None, None  # TODO: (A, b)


track = load_track("test-circuit")
n = len(track)
A, b = None, None  # TODO
shipped = None  # TODO
objective_centre, direct_centre = None, None  # TODO
objective_line, direct_line = None, None  # TODO
largest_gap = None  # TODO
objective_reduction = None  # TODO
curvature_reduction = None  # TODO
same_story = "TODO"

## Exercise 4 - QR, and why it gives the same answer

Unit 8 taught that with `Q, R = np.linalg.qr(A)` (Q is 2n × m with orthonormal columns, R is m × m upper triangular), ‖A c − b‖² = ‖R c − Qᵀb‖² + const for **every** c, the constant being ‖b‖² − ‖Qᵀb‖².
Check it on Exercise 3's `A` and `b`.

Define:

- `Q`, `R`: the factors;
- `q_shape`, `r_shape`: their shapes (tuples);
- `orthonormal_error`: the largest |G[i, k] − 1| for i = k and |G[i, k]| for i ≠ k, over the entries of G = QᵀQ (m × m; QᵀQ is the identity when the columns are orthonormal);
- `below_diagonal`: the largest |R[i, k]| over i > k (R's entries below its diagonal);
- `test_controls`: a list of three control vectors: all zeros, `shipped`, and `3.0 * np.sin(np.arange(m))` (any c will do);
- `differences`: a list of ‖A c − b‖² − ‖R c − Qᵀb‖² for the three, in that order;
- `outside_part`: ‖b‖² − ‖Qᵀb‖²;
- `largest_relative_mismatch`: the largest |difference − `outside_part`| / ‖b‖² over the three;
- `lstsq_gap`: the largest |entry| of the difference between the unbounded solutions `np.linalg.lstsq(A, b, rcond=None)[0]` and `np.linalg.lstsq(R, Q.T @ b, rcond=None)[0]`;
- `why_r_is_enough`: one or two sentences: the constant does not depend on c, so what does that say about the c that minimises either side, with or without bounds, and why is the R problem cheaper?

In [ ]:
Q, R = None, None  # TODO
q_shape, r_shape = None, None  # TODO
orthonormal_error = None  # TODO
below_diagonal = None  # TODO
test_controls = []  # TODO
differences = []  # TODO
outside_part = None  # TODO
largest_relative_mismatch = None  # TODO
lstsq_gap = None  # TODO
why_r_is_enough = "TODO"

## Exercise 5 - Solve a line on `test-circuit-seed1`

Write `solve_line(track, margin=2.5, pin=30)`, the lesson's solver, returning the control values as a NumPy array:

1. the basis `spline_basis(n, 10)`, the normals, and `smoothness_system(track.x, track.y, basis)`;
2. the bounds: control value k's support is the waypoints where column k of the basis is > 0; its bound h_k is the smallest `track.half_width[i] - margin` over its support, except that h_k = 0 when its support contains any of the `pin` waypoints at each end of the lap (indices 0 … pin − 1 and n − pin … n − 1);
3. `Q, R = np.linalg.qr(A)`, then `lsq_linear(R, Q.T @ b, bounds=(-h - 1e-12, h + 1e-12), method="bvls")`, and return its `.x` (the 1e-12 keeps every lower bound strictly below its upper bound, as `lsq_linear` requires).

Solve `test-circuit-seed1` (`seed1 = load_track("test-circuit-seed1")`).
Then solve `test-circuit` and compare with the reference agent's offsets: build an agent from the waypoints, `Unit09(track.waypoints, None)` (Exercise 3's class), whose `.offsets` is the list of its per-waypoint offsets.

Define:

- `solve_line`: the function;
- `seed1_control`, `seed1_offsets`: the control values and the offsets `spline_basis(len(seed1), 10) @ seed1_control`;
- `inside_margin`: `True` when every |offset| ≤ half-width − 2.5 + 1e-9;
- `pinned_start`: `True` when every |offset| on the 30 waypoints at each end of the lap is ≤ 1e-9;
- `on_bound_count`: the number of control values with h_k > 0 and |c_k| within 1e-9 of h_k;
- `centre_numbers`, `line_numbers`: `line_summary` of the centreline and of the line's points;
- `curvature_change`: `line_numbers[0] / centre_numbers[0] - 1`; `shorter_by`: the centreline's lap minus the line's (m); `tightest_change`: the line's tightest radius minus the centreline's (m);
- `reference_gap`: the largest |difference| between your `test-circuit` offsets and the reference agent's `.offsets` (m);
- `fig`, `ax`: the offsets of `test-circuit-seed1`'s line against `seed1.s` (the distance along the centreline), with ± (half-width − 2.5) drawn for comparison, axis labels and a title.

In [ ]:
def solve_line(track, margin=2.5, pin=30):
    return None  # TODO: the control values


seed1 = load_track("test-circuit-seed1")
seed1_control, seed1_offsets = None, None  # TODO
inside_margin = None  # TODO
pinned_start = None  # TODO
on_bound_count = None  # TODO
centre_numbers, line_numbers = None, None  # TODO
curvature_change, shorter_by, tightest_change = None, None, None  # TODO
reference_gap = None  # TODO
fig, ax = None, None  # TODO

## Exercise 6 - Following a line, and the clean rule

**The `REACHED_RADIUS` trap.**
Use `test-circuit` and the reference's offsets (`np.array(agent.offsets)` for the agent of Exercise 5, or `spline_basis(n, 10) @ shipped`).
Count the line points that have **no** centreline waypoint within `REACHED_RADIUS` (the smallest distance from the line point to any of the n centreline waypoints is ≥ 3.0 m).

**The clean rule.**
racekit's telemetry `offset` is the car's signed distance from the **centreline** (left positive); a row with progress p is at waypoint (p + 1) mod n.
Write:

- `line_error(result, offsets)`: the largest |`offset` − offsets[(p + 1) mod n]| over the driving rows (`held == 0`), n = `len(offsets)`;
- `clean(result, offsets)`: finished, 0 collisions, and `line_error` at most `CLEAN_ERROR`.

Race the reference with `repeat_races(UNIT_09, "test-circuit", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)`.

Define:

- `line_error`, `clean`: the functions;
- `unreachable`: the count above, and `unreachable_fraction`: it divided by n;
- `reference_results`: the `RepeatResult` (Exercise 9 reuses it);
- `errors_from_line`: the three races' `line_error` with the reference's offsets (m), in seed order;
- `errors_from_centre`: the three races' `line_error` with offsets of all zeros (the max |offset from the centreline|);
- `unit_7_verdicts`: for each race, `True` when it finished with 0 collisions and its max |offset from the centreline| was at most 2 m (Unit 7's rule);
- `line_verdicts`: for each race, `clean(result, reference offsets)`;
- `explain`: two sentences: why would an agent that advanced its index on the **centreline's** waypoints get stuck on this line, and why is Unit 7's rule the wrong rule for a line race?

In [ ]:
def line_error(result, offsets):
    return None  # TODO


def clean(result, offsets):
    return None  # TODO


unreachable, unreachable_fraction = None, None  # TODO
reference_results = None  # TODO
errors_from_line = []  # TODO
errors_from_centre = []  # TODO
unit_7_verdicts = []  # TODO
line_verdicts = []  # TODO
explain = "TODO"

## Exercise 7 - The honest comparison on `test-circuit-seed1`

Is the line's gain at the same f a `test-circuit` accident?
Race both agents on `test-circuit-seed1` at f = 0.54, with that track's measured grip:

- the line: `variant(UNIT_09, "seed1-line", LINE_CONTROL=tuple(float(v) for v in seed1_control), MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54)`;
- the centreline: `variant(UNIT_08, "seed1-centre", MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54)`;

each with `repeat_races` on `"test-circuit-seed1"`, seeds 0, 1 and 2 (`n=3, seeds=(0, 1, 2)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.

Define:

- `line_results`, `centre_results`: the two `RepeatResult` objects;
- `all_clean`: `True` when all six races are clean by Exercise 6's `clean` (the line's races against `seed1_offsets`, the centreline's against zeros);
- `medians`, `spreads`: dicts `{"line": ..., "centre": ...}` of the median race time and the spread (s);
- `line_better`: `noise_aware_better(line_results.times, centre_results.times)`;
- `gain_s`: the centreline's median minus the line's (s);
- `compared_with_test_circuit`: one sentence: the lesson measured about 25 s on `test-circuit` at the same f; is this track's gain similar, and why might the gain differ from track to track?

In [ ]:
line_results, centre_results = None, None  # TODO
all_clean = None  # TODO
medians, spreads = {}, {}  # TODO
line_better = None  # TODO
gain_s = None  # TODO
compared_with_test_circuit = "TODO"

## Exercise 8 - Challenge: a late-apex window

The lesson's local widening opened a **symmetric** window, 60 waypoints either side of every point tighter than 60 m radius.
Racing drivers prefer a **late apex**: turn in later, and use the road on the way **out** of the corner, where the car is accelerating.
Try a window with more waypoints after the tight point than before, on `monza-like` (where the symmetric window races clean).

Write:

- `window_mask(x, y, before, after, radius=60.0)`: a boolean NumPy array, `True` at every waypoint k with i − before ≤ k ≤ i + after (mod n) for some waypoint i whose |Menger curvature| is > 1 / radius ("after" is the direction of travel);
- `local_line(track, near, margin=2.5, pin=30)`: Exercise 5's solver with one more rule: h_k = 0 also when control value k's support contains any waypoint where `near` is `False`.

Solve two lines on `monza-like`: symmetric, `window_mask(..., 60, 60)`, and late, `window_mask(..., 30, 90)`.
Race each as `variant(UNIT_09, ..., LINE_CONTROL=..., MU_G=MEASURED_MU_G["monza-like"], MU_FRACTION=0.60)` with `repeat_races` on `"monza-like"`, seeds 0 and 1 (`n=2, seeds=(0, 1)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=900.0`.

Define:

- `window_mask`, `local_line`: the functions;
- `symmetric_control`, `late_control`: the two lines' control values;
- `summaries`: a dict `{"symmetric": ..., "late": ...}` of each line's `line_summary` on its points;
- `symmetric_results`, `late_results`: the two `RepeatResult` objects;
- `windows_clean`: `True` when all four races are clean by Exercise 6's `clean`, each against its own line's offsets;
- `window_medians`: a dict `{"symmetric": ..., "late": ...}` of the median race times (s);
- `late_better`: `noise_aware_better(late_results.times, symmetric_results.times)`;
- `why_late`: two sentences: which line is smoother by Σκ²Δs, which is faster, and what does that say about using the smoothness objective alone to choose a line?

In [ ]:
def window_mask(x, y, before, after, radius=60.0):
    return None  # TODO


def local_line(track, near, margin=2.5, pin=30):
    return None  # TODO


monza = load_track("monza-like")
symmetric_control, late_control = None, None  # TODO
summaries = {}  # TODO
symmetric_results, late_results = None, None  # TODO
windows_clean = None  # TODO
window_medians = {}  # TODO
late_better = None  # TODO
why_late = "TODO"

## Exercise 9 - Challenge: a 2.0 m margin

The line keeps 2.5 m from each edge.
A narrower margin gives the line more road, so it can be straighter: is it worth the risk?

Solve `test-circuit`'s line with `solve_line(track, margin=2.0)` and race it with the reference's grip and f, `variant(UNIT_09, "margin-2.0", LINE_CONTROL=tuple(float(v) for v in narrow_control))`, with `repeat_races` on `"test-circuit"`, seeds 0, 1 and 2, `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.
Compare with Exercise 6's `reference_results` (the 2.5 m margin, the same seeds).

The car's **room to the edge** on a driving row is the half-width at its waypoint minus its distance from the centreline: `track.half_width[(p + 1) % n] - abs(offset)`.

Define:

- `narrow_control`, `narrow_offsets`: the 2.0 m line's control values and offsets;
- `narrow_numbers`: its `line_summary`;
- `narrow_results`: the `RepeatResult`;
- `narrow_clean`: `True` when its three races are clean by Exercise 6's `clean`;
- `margin_medians`: a dict from margin (2.5 and 2.0) to the median race time (s);
- `time_gained`: `margin_medians[2.5] - margin_medians[2.0]` (s);
- `beats_the_noise`: `noise_aware_better(narrow_results.times, reference_results.times)`;
- `closest_to_edge`: a dict from margin to the smallest room to the edge over the driving rows of its three races (m);
- `worth_it`: two or three sentences: is the narrower margin worth its risk? Weigh the time it gains against the room it gives up, and say what CARLA, whose car and track are not racekit's, would add to the risk.

In [ ]:
narrow_control, narrow_offsets = None, None  # TODO
narrow_numbers = None  # TODO
narrow_results = None  # TODO
narrow_clean = None  # TODO
margin_medians = {}  # TODO
time_gained = None  # TODO
beats_the_noise = None  # TODO
closest_to_edge = {}  # TODO
worth_it = "TODO"